In [1]:
# pip install boto3

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.functions import monotonically_increasing_id

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("dim_bank") \
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4"
    ) \
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    ) \
    .config(
        "spark.hadoop.fs.s3a.aws.profile",
        "default"
    ) \
    .config("spark.driver.memory", "10g") \
    .config("spark.driver.memoryOverhead", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

In [3]:
spark.sparkContext._jsc.hadoopConfiguration().set(
    "fs.s3a.aws.credentials.provider",
    "com.amazonaws.auth.profile.ProfileCredentialsProvider"
)

spark.sparkContext._jsc.hadoopConfiguration().set(
    "fs.s3a.aws.profile",
    "default"
)

In [4]:
transactions = spark.read.parquet(
    "s3a://datapath-buckets/bank_datasets/curated/transactions/"
)

accounts = spark.read.parquet(
    "s3a://datapath-buckets/bank_datasets/curated/accounts/"
)

In [5]:
transactions.show(5)
accounts.show(5)

+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+-----------+------------+----+-----+
|          Timestamp|From Bank|Account_From|To Bank|Account_To|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|minute_part|day_of_month|year|month|
+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+-----------+------------+----+-----+
|2022-09-01 00:16:00|        1|   8000EC1E0|      1| 8000EC1E0|          11.86|         US Dollar|      11.86|       US Dollar|  Reinvestment|            0|         16|           1|2022|    9|
|2022-09-01 00:04:00|        1|   8000F4510|  11813| 8011305D0|           9.82|         US Dollar|       9.82|       US Dollar|   Credit Card|            0|          4|           1|2022|    9|
|2022-09-01 00:11:00|       12|   8

In [6]:
transactions.printSchema()
accounts.printSchema()

root
 |-- Timestamp: timestamp (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- Account_From: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Account_To: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)
 |-- minute_part: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

root
 |-- Bank Name: string (nullable = true)
 |-- Bank ID: integer (nullable = true)
 |-- Account Number: string (nullable = true)
 |-- Entity ID: string (nullable = true)
 |-- Entity Name: string (nullable = true)



In [7]:
df_resultado = transactions.join(
    accounts,
    transactions["Account_From"] == accounts["Account Number"],
    how="inner"
)

df_resultado.show(5)

+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+-----------+------------+----+-----+------------+-------+--------------+---------+-----------------+
|          Timestamp|From Bank|Account_From|To Bank|Account_To|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|minute_part|day_of_month|year|month|   Bank Name|Bank ID|Account Number|Entity ID|      Entity Name|
+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+-----------+------------+----+-----+------------+-------+--------------+---------+-----------------+
|2022-09-01 00:01:00|       70|   1004286A8|     20| 800103610|         439.15|              Euro|     439.15|            Euro|        Cheque|            0|          1|           1|2022|    9|Oasis Thrift|     70|     1004286A8|8

In [8]:
df_resultado = df_resultado.withColumnRenamed("From Bank", "bank_id") 

In [9]:
bancos_unicos = df_resultado.select("bank_id", "Bank Name").distinct()
bancos_unicos.show(20)
print(bancos_unicos.count())
print(bancos_unicos.select("bank_id").distinct().count())

+-------+--------------------+
|bank_id|           Bank Name|
+-------+--------------------+
|     12|National Bank of ...|
|   1467|  Plandor Trust Bank|
|    513|National Bank of ...|
|   3608|National Bank of ...|
|   3701|First Bank of Lin...|
|   3291| Desert Credit Union|
|   1729|     Willows Bancorp|
|   3210|Fieldstone Trust ...|
|   2843|       Sappo Bancorp|
|  21174|    Bank of the West|
|   3717|Sappo Cooperative...|
|  31422|      Bank of Tuscon|
|   1411|   Bank of the South|
|  11474| First Bank of Butte|
|  21611|     The Pine Thrift|
|  31786|National Bank of ...|
|   1686|Brownstone Credit...|
|  32013|Savings Bank of O...|
|  22164|Spruce Cooperativ...|
|   3641|National Bank of ...|
+-------+--------------------+
only showing top 20 rows

30478
30470


In [10]:
from pyspark.sql import functions as F

bancos_unicos.groupBy("bank_id") \
    .agg(F.count("Bank Name").alias("num_nombres")) \
    .filter(F.col("num_nombres") > 1) \
    .show()

+-------+-----------+
|bank_id|num_nombres|
+-------+-----------+
|  27444|          2|
|   1490|          2|
|  27755|          2|
| 142574|          2|
|  28248|          2|
| 221731|          2|
| 138832|          2|
|  13858|          2|
+-------+-----------+



In [11]:
ids_conflicto = [27444, 1490, 27755, 142574, 28248, 221731, 138832, 13858]

bancos_unicos.filter(F.col("bank_id").isin(ids_conflicto)) \
    .orderBy("bank_id") \
    .show(20, truncate=False)

+-------+-------------------------+
|bank_id|Bank Name                |
+-------+-------------------------+
|1490   |Italy Bank #95           |
|1490   |Germany Bank #945        |
|13858  |Italy Bank #97           |
|13858  |Germany Bank #908        |
|27444  |Savings Bank of Fairfield|
|27444  |National Bank of Danbury |
|27755  |Australia Bank #44       |
|27755  |Australia Bank #47       |
|28248  |Australia Bank #47       |
|28248  |Australia Bank #44       |
|138832 |Germany Bank #908        |
|138832 |Italy Bank #97           |
|142574 |Germany Bank #945        |
|142574 |Italy Bank #95           |
|221731 |National Bank of Danbury |
|221731 |Savings Bank of Fairfield|
+-------+-------------------------+



In [12]:
from pyspark.sql import Window

w = Window.partitionBy("bank_id").orderBy("Bank Name")
dim_bank = bancos_unicos.withColumn("rn", F.row_number().over(w)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

In [13]:
dim_bank.show(20, truncate=False)

+-------+-------------------------+
|bank_id|Bank Name                |
+-------+-------------------------+
|1      |Arbor Savings Bank       |
|3      |China Bank #14           |
|4      |China Bank #3            |
|5      |UK Bank #0               |
|6      |India Bank #0            |
|7      |India Bank #25           |
|8      |Canada Bank #12          |
|9      |Russia Bank #16          |
|10     |National Bank of Laramie |
|11     |Finland Bank #0          |
|12     |National Bank of the East|
|14     |China Bank #13           |
|15     |Japan Bank #0            |
|16     |India Bank #32           |
|19     |Australia Bank #3        |
|20     |China Bank #6            |
|21     |Japan Bank #18           |
|22     |Germany Bank #65         |
|23     |Germany Bank #18         |
|24     |Japan Bank #41           |
+-------+-------------------------+
only showing top 20 rows



In [14]:
dim_bank.write \
    .mode("overwrite") \
    .parquet("s3a://datapath-buckets/bank_datasets/curated/dimensional_model/dim_bank/")